## Setup Genetic Algorithm

### Imports

In [3]:
import random
import csv
import numpy as np
from deap import base, creator, tools

#### Initialise standard values and Setup

In [ ]:
T = 30  # Max days of schedule 
M = 100 # Number of engines
MAX_DAILY_PENALTY = 250 # Daily downtime penalty 


TEAMS = {
    "T1": "A",
    "T2": "B",
    "T3": "A",
    "T4": "B",
}

NUM_TEAMS = len(TEAMS)
TEAM_NAMES = list(TEAMS.keys())
TEAM_TYPES = list(TEAMS.values())

def load_rul_predictions(filepath):
    rul_dict = {}
    with open(filepath, "r") as f:
        reader = csv.DictReader(f, delimiter=";")
        for row in reader:
            rul_dict[int(row["id"])] = int(row["RUL"])
    return rul_dict

{1: 135,
 2: 125,
 3: 63,
 4: 100,
 5: 103,
 6: 122,
 7: 106,
 8: 90,
 9: 121,
 10: 67,
 11: 101,
 12: 89,
 13: 87,
 14: 122,
 15: 114,
 16: 101,
 17: 52,
 18: 33,
 19: 84,
 20: 10,
 21: 63,
 22: 141,
 23: 119,
 24: 26,
 25: 173,
 26: 128,
 27: 70,
 28: 96,
 29: 96,
 30: 87,
 31: 14,
 32: 54,
 33: 128,
 34: 8,
 35: 8,
 36: 24,
 37: 21,
 38: 58,
 39: 144,
 40: 29,
 41: 23,
 42: 13,
 43: 67,
 44: 146,
 45: 100,
 46: 53,
 47: 130,
 48: 151,
 49: 14,
 50: 100,
 51: 106,
 52: 34,
 53: 34,
 54: 126,
 55: 174,
 56: 18,
 57: 102,
 58: 38,
 59: 113,
 60: 112,
 61: 23,
 62: 54,
 63: 75,
 64: 24,
 65: 152,
 66: 18,
 67: 174,
 68: 13,
 69: 130,
 70: 90,
 71: 130,
 72: 59,
 73: 113,
 74: 115,
 75: 115,
 76: 3,
 77: 27,
 78: 165,
 79: 82,
 80: 84,
 81: 6,
 82: 11,
 83: 182,
 84: 53,
 85: 142,
 86: 113,
 87: 126,
 88: 117,
 89: 111,
 90: 28,
 91: 29,
 92: 24,
 93: 51,
 94: 55,
 95: 143,
 96: 140,
 97: 109,
 98: 87,
 99: 127,
 100: 24}

#### Duration functions

In [9]:
def mu_A(j):
    """Maintenance duration for team A"""
    if (1 <= j <= 20) or (81 <= j <= 100):
        return 5
    elif 21 <= j <= 55:
        return 3
    elif 56 <= j <= 80:
        return 4
    else:
        raise ValueError(f"Engine index {j} out of range [1, 100]")

def mu_B(j):
    """Maintenance duration for team B"""
    if 1 <= j <= 25:
        return mu_A(j) - 1
    elif 26 <= j <= 70:
        return mu_A(j) + 3
    elif 71 <= j <= 100:
        return mu_A(j) + 2
    else:
        raise ValueError(f"Engine index {j} out of range [1, 100]")

def maintenance_duration(team_index, engine_id):
    """Maintenance duration for a team on a given engine"""
    if TEAM_TYPES[team_index] == "A":
        return mu_A(engine_id)
    else:
        return mu_B(engine_id)

#### Penalty functions

In [10]:
def cost_coefficient(j):
    """Penalty cost coefficient for engine j"""
    if 1 <= j <= 25:
        return 4
    elif 26 <= j <= 45:
        return 2
    elif 46 <= j <= 75:
        return 5
    elif 76 <= j <= 100:
        return 6
    else:
        raise ValueError(f"Engine index {j} out of range [1, 100]")


def penalty_cost(engine_id, safety_due_date, maintenance_end_day):
    """
    Calculate the total penalty cost for an engine
    engine_id:          engine index (1-100)
    safety_due_date:    last day the engine works (ts)
    maintenance_end_day: day maintenance finishes (None if not maintained)
    """
    cj = cost_coefficient(engine_id)
    total_penalty = 0
    
    if maintenance_end_day is None: # If engine is never repaired
        # Penalty for every day after ts until end of schedule
        for t in range(safety_due_date + 1, T + 1): 
            days_late = t - safety_due_date
            daily_cost = min(cj * days_late ** 2, MAX_DAILY_PENALTY) # cost cannot be higher than the max penalty
            total_penalty += daily_cost
    else:
        # Penalty for days between due date and maintenance completion
        for t in range(safety_due_date + 1, maintenance_end_day + 1):
            days_late = t - safety_due_date
            daily_cost = min(cj * days_late ** 2, MAX_DAILY_PENALTY) # cost cannot be higher than the max penalty
            total_penalty += daily_cost
    
    return total_penalty

In [13]:
def get_urgent_engines(rul_dict):
    """Sorted list of engine IDs with their RUL if RUL < T"""
    return sorted([engine_id for engine_id, rul in rul_dict.items() if rul < T])

 

 # TEST code van CLAUDE
if __name__ == "__main__":
    # Load consultancy predictions
    rul_predictions = load_rul_predictions('RUL_consultancy_predictions_A3.csv')
    urgent_engines = get_urgent_engines(rul_predictions)
    
    print(f"Planning horizon: T = {T} days")
    print(f"Total engines: {M}")
    print(f"Engines needing maintenance (due date < {T}): {len(urgent_engines)}")
    print(f"\n{'Engine':>7} {'RUL':>4} {'DueDate':>8} {'muA':>4} {'muB':>4} {'cj':>4}")
    print("-" * 40)
    for eng_id in urgent_engines:
        rul = rul_predictions[eng_id]
        print(f"{eng_id:>7} {rul:>4} {rul:>8} {mu_A(eng_id):>4} {mu_B(eng_id):>4} {cost_coefficient(eng_id):>4}")
    
    # Test penalty calculation: engine 76 (due day 3), not maintained
    p = penalty_cost(76, 3, None)
    print(f"\nExample: Engine 76, due day 3, NOT maintained -> penalty = {p}")
    
    # Test penalty calculation: engine 76, maintained by day 5
    p2 = penalty_cost(76, 3, 5)
    print(f"Example: Engine 76, due day 3, maintained by day 5 -> penalty = {p2}")
 

Planning horizon: T = 30 days
Total engines: 100
Engines needing maintenance (due date < 30): 24

 Engine  RUL  DueDate  muA  muB   cj
----------------------------------------
     20   10       10    5    4    4
     24   26       26    3    2    4
     31   14       14    3    6    2
     34    8        8    3    6    2
     35    8        8    3    6    2
     36   24       24    3    6    2
     37   21       21    3    6    2
     40   29       29    3    6    2
     41   23       23    3    6    2
     42   13       13    3    6    2
     49   14       14    3    6    5
     56   18       18    4    7    5
     61   23       23    4    7    5
     64   24       24    4    7    5
     66   18       18    4    7    5
     68   13       13    4    7    5
     76    3        3    4    6    6
     77   27       27    4    6    6
     81    6        6    5    7    6
     82   11       11    5    7    6
     90   28       28    5    7    6
     91   29       29    5    7    6
     92   

## Population initialisation (Nog allemaal van Claude, niet gecontroleerd)

In [ ]:
# =============================================================================
# STEP 2: Individual Representation, Decoder & Initialization
# =============================================================================
# 
# Representation:
#   An individual is a list of N_urgent integers, one per urgent engine.
#   Each gene value is 0-4:
#     0 = engine is not scheduled for maintenance
#     1 = assigned to team T1 (type A)
#     2 = assigned to team T2 (type B)
#     3 = assigned to team T3 (type A)
#     4 = assigned to team T4 (type B)
#
# Decoder:
#   Converts an individual into a concrete schedule. For each team, the
#   assigned engines are sorted by due date (earliest first) and scheduled
#   sequentially from day 1. If a job doesn't fit within T=30, it is skipped.
 
def decode_individual(individual, urgent_engines, rul_dict):
    """
    Decode an individual (list of team assignments) into a maintenance schedule.
    
    Parameters:
        individual:     list of integers (0-4), one per urgent engine
        urgent_engines: list of urgent engine IDs
        rul_dict:       dictionary {engine_id: RUL}
    
    Returns:
        schedule: dict with team_index (0-3) as keys, each value is a list of
                  tuples (engine_id, start_day, end_day)
        unscheduled: list of engine IDs that were assigned but didn't fit
    """
    # Group engines by team assignment
    team_jobs = {team_idx: [] for team_idx in range(NUM_TEAMS)}
    
    for i, team_assignment in enumerate(individual):
        if team_assignment == 0:
            continue  # engine not scheduled
        engine_id = urgent_engines[i]
        due_date = rul_dict[engine_id]
        team_idx = team_assignment - 1  # convert 1-4 to 0-3
        team_jobs[team_idx].append((engine_id, due_date))
    
    # For each team, sort by due date (earliest first) and schedule sequentially
    schedule = {team_idx: [] for team_idx in range(NUM_TEAMS)}
    unscheduled = []
    
    for team_idx in range(NUM_TEAMS):
        # Sort assigned engines by due date (most urgent first)
        jobs = sorted(team_jobs[team_idx], key=lambda x: x[1])
        current_day = 1
        
        for engine_id, due_date in jobs:
            duration = maintenance_duration(team_idx, engine_id)
            start_day = current_day
            end_day = start_day + duration - 1
            
            # Check if maintenance fits within planning horizon
            if end_day <= T:
                schedule[team_idx].append((engine_id, start_day, end_day))
                current_day = end_day + 1  # next job starts after this one
            else:
                unscheduled.append(engine_id)
    
    return schedule, unscheduled
 
def evaluate(individual, urgent_engines, rul_dict):
    """
    Fitness function: compute total penalty cost for a maintenance schedule.
    
    Returns a tuple (required by DEAP for single-objective optimization).
    """
    schedule, unscheduled = decode_individual(individual, urgent_engines, rul_dict)
    
    # Build a lookup: engine_id -> end_day of maintenance
    maintenance_end = {}
    for team_idx in range(NUM_TEAMS):
        for engine_id, start_day, end_day in schedule[team_idx]:
            maintenance_end[engine_id] = end_day
    
    # Calculate total penalty cost for all urgent engines
    total_cost = 0
    for engine_id in urgent_engines:
        due_date = rul_dict[engine_id]
        end_day = maintenance_end.get(engine_id, None)
        total_cost += penalty_cost(engine_id, due_date, end_day)
    
    return (total_cost,)  # DEAP requires a tuple
 
# --- DEAP type setup ---
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)
 
def setup_toolbox(urgent_engines, rul_dict):
    """
    Create and configure the DEAP toolbox.
    
    Parameters:
        urgent_engines: list of urgent engine IDs
        rul_dict:       dictionary {engine_id: RUL}
    
    Returns:
        Configured DEAP Toolbox
    """
    n_urgent = len(urgent_engines)
    toolbox = base.Toolbox()
    
    # Each gene is an integer 0-4 (0=no maintenance, 1-4=team assignment)
    toolbox.register("gene", random.randint, 0, NUM_TEAMS)
    
    # An individual is a list of n_urgent genes
    toolbox.register(
        "individual",
        tools.initRepeat,
        creator.Individual,
        toolbox.gene,
        n=n_urgent,
    )
    
    # A population is a list of individuals
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    
    # Register the evaluation function with fixed parameters
    toolbox.register("evaluate", evaluate, urgent_engines=urgent_engines, rul_dict=rul_dict)
    
    return toolbox
 
 
# =============================================================================
# Quick test of Step 1 & 2
# =============================================================================
if __name__ == "__main__":
    # Load consultancy predictions
    rul_predictions = load_rul_predictions(
        "/mnt/user-data/uploads/RUL_consultancy_predictions_A3.csv"
    )
    urgent_engines = get_urgent_engines(rul_predictions)
    
    print(f"Planning horizon: T = {T} days")
    print(f"Total engines: {M}")
    print(f"Engines needing maintenance (due date < {T}): {len(urgent_engines)}")
    print(f"\n{'Engine':>7} {'RUL':>4} {'DueDate':>8} {'muA':>4} {'muB':>4} {'cj':>4}")
    print("-" * 40)
    for eng_id in urgent_engines:
        rul = rul_predictions[eng_id]
        print(f"{eng_id:>7} {rul:>4} {rul:>8} {mu_A(eng_id):>4} {mu_B(eng_id):>4} {cost_coefficient(eng_id):>4}")
    
    # --- Test Step 2: create a random individual and decode it ---
    print("\n" + "=" * 50)
    print("STEP 2 TEST: Random individual")
    print("=" * 50)
    
    toolbox = setup_toolbox(urgent_engines, rul_predictions)
    
    random.seed(42)
    ind = toolbox.individual()
    print(f"\nIndividual (team assignments): {list(ind)}")
    print(f"Mapping:")
    for i, eng_id in enumerate(urgent_engines):
        team = ind[i]
        label = f"Team T{team}" if team > 0 else "Not scheduled"
        print(f"  Engine {eng_id:>3} -> {label}")
    
    # Decode and show schedule
    schedule, unscheduled = decode_individual(ind, urgent_engines, rul_predictions)
    print(f"\nDecoded schedule:")
    for team_idx in range(NUM_TEAMS):
        team_name = TEAM_NAMES[team_idx]
        team_type = TEAM_TYPES[team_idx]
        jobs = schedule[team_idx]
        print(f"  {team_name} (type {team_type}):")
        if not jobs:
            print(f"    No jobs assigned")
        for engine_id, start, end in jobs:
            print(f"    Engine {engine_id:>3}: day {start:>2} -> day {end:>2}")
    
    if unscheduled:
        print(f"\n  Unscheduled (didn't fit): {unscheduled}")
    
    # Evaluate fitness
    fitness = toolbox.evaluate(ind)
    print(f"\nTotal penalty cost: {fitness[0]}")
 